In [1]:
# @title
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
!pip install gfpgan>=1.3.5
!pip install basicsr>=1.3.3.11
!pip install facexlib>=0.2.0.3
!pip install gfpgan>=0.2.1
!pip install -r requirements.txt
!python setup.py develop

Cloning into 'Real-ESRGAN'...
remote: Enumerating objects: 759, done.
remote: Total 759 (delta 0), reused 0 (delta 0), pack-reused 759 (from 1)
Receiving objects: 100% (759/759), 5.39 MiB | 24.19 MiB/s, done.
Resolving deltas: 100% (408/408), done.
/content/Real-ESRGAN
/usr/local/lib/python3.12/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
!!

        ********************************************************************************
        Requirements should be satisfied by a PEP 517 installer.
        If you are using pip, you can try `pip install --use-pep517`.
        ********************************************************************************

!!
  dist.fetch_build_eggs(dist.setup_requires)
running develop
/usr/local/lib/python3.12/dist-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ************************************************

In [2]:
import os
os.makedirs('weights', exist_ok=True)
!wget https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P ./weights
!wget https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth -P ./weights

--2026-07-30 09:38:47--  https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/387326890/08f0e941-ebb7-48f0-9d6a-73e87b710e7e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-30T10%3A12%3A10Z&rscd=attachment%3B+filename%3DRealESRGAN_x4plus.pth&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-07-30T09%3A11%3A18Z&ske=2026-07-30T10%3A12%3A10Z&sks=b&skv=2018-11-09&sig=kt0tGtt3spmzEkFbMs66Vke2ycX1iVHp43880sjvqSA%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4NTQwNjEyNywibmJmIjoxNzg1NDA0MzI3LCJwYXRoIjoicmVsZWFzZWFzc2V0cH

In [3]:
# @title
import os
import sys

# Traceback'te belirtilen dosya yolu
DEGRADATIONS_FILE = "/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py"

try:
    with open(DEGRADATIONS_FILE, 'r') as f:
        content = f.read()

    # Yama 1: Hatalı import satırını devre dışı bırak
    old_import_line = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
    new_import_line = "# from torchvision.transforms.functional_tensor import rgb_to_grayscale # YAMA: Colab uyumluluğu"
    content = content.replace(old_import_line, new_import_line)

    # Yama 2: Eğer içerideki kod bu fonksiyonu kullanıyorsa onu da devre dışı bırak (örn. gfpgan kullanıyorsa)
    # Bu adımı sadece, import'u devre dışı bıraktıktan sonra başka bir hata alırsak deneriz.
    # Şu an sadece import'u kapattık. Dosya içeriğini tekrar yazıyoruz.

    with open(DEGRADATIONS_FILE, 'w') as f:
        f.write(content)

    print(f"✅ basicsr import yaması uygulandı: {DEGRADATIONS_FILE}")
    print("⚠️ Hata devam ederse, içerideki fonksiyon çağrısını da düzenlememiz gerekecek.")

except FileNotFoundError:
    print(f"❌ Hata: Dosya yolu bulunamadı. Kurulumdan sonra bu dosyayı tekrar arayın.")
except Exception as e:
    print(f"❌ Dosya düzenlenirken hata oluştu: {e}")

✅ basicsr import yaması uygulandı: /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py
⚠️ Hata devam ederse, içerideki fonksiyon çağrısını da düzenlememiz gerekecek.


In [4]:
# @title
from basicsr.archs.rrdbnet_arch import RRDBNet
import cv2

from realesrgan import RealESRGANer
from realesrgan.archs.srvgg_arch import SRVGGNetCompact
from gfpgan import GFPGANer
import tempfile
import base64
import numpy as np
import io
from PIL import Image

In [5]:
# @title
# GPU ve VRAM Optimizasyonu (A100 ve CPU Uyumlu)
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# GPU varsa FP16 (yarı hassasiyet) aktif edilir, CPU'da hata vermemesi için False yapılır.
use_half = torch.cuda.is_available()

model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
netscale = 4
model_path = os.path.join("weights", "RealESRGAN_x4plus.pth")
upsampler = RealESRGANer(
            scale=netscale,
            model_path=model_path,
            model=model,
            tile=0,
            tile_pad=10,
            pre_pad=0,
            half=use_half)

gfpgan_path = os.path.join("weights", "GFPGANv1.4.pth")
restorer = GFPGANer(
            model_path=gfpgan_path,
            upscale=netscale,
            arch='clean',
            channel_multiplier=2,
            bg_upsampler=upsampler)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Downloading: "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth" to /content/Real-ESRGAN/gfpgan/weights/detection_Resnet50_Final.pth



100%|██████████| 104M/104M [00:00<00:00, 468MB/s] 


Downloading: "https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth" to /content/Real-ESRGAN/gfpgan/weights/parsing_parsenet.pth



100%|██████████| 81.4M/81.4M [00:00<00:00, 508MB/s]


In [11]:
#@title 🐺 Orioninsist | Profesyonel Yapay Zeka Upscaler (300 DPI Entegrasyonlu)
#@markdown Bu araç, model resimlerinizi veya tişört tasarımlarınızı yapay zeka ile 4-8 kat büyüterek 300 DPI baskı kalitesine ulaştırır.
#@markdown Kodları gizlemek için hücre başlığına çift tıklayabilir ve işlemleri doğrudan form üzerinden yürütebilirsiniz.
#@markdown ---

scale = 4 #@param {type:"slider", min:2, max:8, step:1}
#@markdown *   *İpucu:* Görselin kaç kat büyütüleceğini seçin (300 DPI baskı için 4 idealdir).

use_face_restore = False #@param {type:"boolean"}
#@markdown *   ⚠️ **ÖNEMLİ KURAL**: Eğer tişört üzerine basılacak grafikleri/logoları büyütüyorsanız bu ayarı kesinlikle **KAPATIN (False)**.
#@markdown *   Manken görsellerini (Freya) büyütürken yüz hatlarının pürüzsüzleşmesi için bu ayarı **AÇIN (True)**.

from google.colab import files
import os
import cv2
import numpy as np
from PIL import Image

def process_and_upscale(image_path, output_path, scale, use_face_restore=False):
    img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        print(f"❌ Hata: {image_path} yüklenemedi.")
        return False

    channels = img.shape[2] if len(img.shape) == 3 else 1
    if channels == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        channels = 3

    has_alpha = (channels == 4)

    try:
        if use_face_restore:
            if has_alpha:
                bgr = img[:, :, :3]
                alpha = img[:, :, 3]

                _, _, restored_bgr = restorer.enhance(
                    bgr,
                    has_aligned=False,
                    only_center_face=False,
                    paste_back=True,
                    weight=0.5
                )

                alpha_bgr = cv2.merge([alpha, alpha, alpha])
                upscaled_alpha_bgr, _ = upsampler.enhance(alpha_bgr, outscale=scale)
                upscaled_alpha = upscaled_alpha_bgr[:, :, 0]

                b, g, r = cv2.split(restored_bgr)
                output = cv2.merge([b, g, r, upscaled_alpha])
            else:
                _, _, output = restorer.enhance(
                    img,
                    has_aligned=False,
                    only_center_face=False,
                    paste_back=True,
                    weight=0.5
                )
        else:
            output, _ = upsampler.enhance(img, outscale=scale)

        # Sonucu kaydet ve 300 DPI meta verisini enjekte et
        if has_alpha:
            pil_img = Image.fromarray(cv2.cvtColor(output, cv2.COLOR_BGRA2RGBA))
            pil_img.save(output_path, dpi=(300, 300))
        else:
            pil_img = Image.fromarray(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
            pil_img.save(output_path, dpi=(300, 300))
        return True
    except RuntimeError as error:
        print('Hata oluştu:', error)
        return False
    except Exception as e:
        print(f"Beklenmedik hata: {e}")
        return False

print("Lütfen bilgisayarınızdan büyütmek istediğiniz görsel(ler)i seçin:")
uploaded = files.upload()

if len(uploaded) == 0:
    print("❌ Hiçbir dosya seçilmedi.")
else:
    for filename in uploaded.keys():
        print("-" * 60)
        print(f"⏳ İşleniyor: {filename} ...")

        name, ext = os.path.splitext(filename)
        output_filename = f"{name}_upscaled{ext}"

        success = process_and_upscale(filename, output_filename, scale, use_face_restore)

        if success:
            print(f"✅ Başarılı! Bilgisayarınıza indiriliyor: {output_filename}")
            files.download(output_filename)


Lütfen bilgisayarınızdan büyütmek istediğiniz görsel(ler)i seçin:


Saving 2026-07-29_rustic-outdoor_geometric-wolf-badge_001 (2)_rembg.png to 2026-07-29_rustic-outdoor_geometric-wolf-badge_001 (2)_rembg (1).png
Saving 2026-07-29_wildlife-nature_watercolor-wolf-floral-top_001 (3)_rembg.png to 2026-07-29_wildlife-nature_watercolor-wolf-floral-top_001 (3)_rembg (1).png
Saving 2026-07-29_wilderness-tee_rustic-pine-forest-wolf_001 (2)_rembg.png to 2026-07-29_wilderness-tee_rustic-pine-forest-wolf_001 (2)_rembg (1).png
Saving 2026-07-29_vintage-wolf_full-moon-retro-graphic-tshirt_001_rembg.png to 2026-07-29_vintage-wolf_full-moon-retro-graphic-tshirt_001_rembg (1).png
------------------------------------------------------------
⏳ İşleniyor: 2026-07-29_rustic-outdoor_geometric-wolf-badge_001 (2)_rembg (1).png ...
✅ Başarılı! Bilgisayarınıza indiriliyor: 2026-07-29_rustic-outdoor_geometric-wolf-badge_001 (2)_rembg (1)_upscaled.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

------------------------------------------------------------
⏳ İşleniyor: 2026-07-29_wildlife-nature_watercolor-wolf-floral-top_001 (3)_rembg (1).png ...
✅ Başarılı! Bilgisayarınıza indiriliyor: 2026-07-29_wildlife-nature_watercolor-wolf-floral-top_001 (3)_rembg (1)_upscaled.png


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

------------------------------------------------------------
⏳ İşleniyor: 2026-07-29_vintage-wolf_full-moon-retro-graphic-tshirt_001_rembg (1).png ...
✅ Başarılı! Bilgisayarınıza indiriliyor: 2026-07-29_vintage-wolf_full-moon-retro-graphic-tshirt_001_rembg (1)_upscaled.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
#@title 🔬 ORIONINSIST — Düzeltilmiş Printify Baskı Kalitesi Analizi
#@markdown Bütün *_upscaled görselleri otomatik inceler.
#@markdown Aynı görselin (1), (2) gibi kopyalarını tekrar saymaz.
#@markdown Sonucu BASKIYA HAZIR / KONTROL ET / YENİDEN İŞLE şeklinde verir.

import os
import re
import glob
import cv2
import numpy as np
from PIL import Image
from IPython.display import display, HTML


# ============================================================
# AYARLAR
# ============================================================

TARGET_DPI = 300
MAX_PRINTIFY_SIZE_MB = 100

BLUR_WARNING = 70
BLUR_FAIL = 35

DARK_WARNING_PERCENT = 35
DARK_FAIL_PERCENT = 60

LIGHT_CLIP_WARNING_PERCENT = 25
DARK_CLIP_WARNING_PERCENT = 25

TRANSPARENT_RESIDUE_WARNING = 3.0
TRANSPARENT_RESIDUE_STRONG = 20.0

HALO_WARNING = 12.0

TINY_DETAIL_WARNING = 22.0

STRUCTURE_WARNING = 12.0
STRUCTURE_FAIL = 25.0


# ============================================================
# YARDIMCI FONKSİYONLAR
# ============================================================

def load_rgba(path):
    """Görseli RGBA olarak açar."""
    with Image.open(path) as image:
        return np.array(image.convert("RGBA"))


def human_size(size_bytes):
    if size_bytes < 1024:
        return f"{size_bytes} B"

    if size_bytes < 1024 ** 2:
        return f"{size_bytes / 1024:.2f} KB"

    return f"{size_bytes / (1024 ** 2):.2f} MB"


def safe_percent(value):
    return max(0.0, min(100.0, float(value)))


def resize_for_analysis(image, max_side=1600):
    """
    Analizi hızlandırmak için geçici kopyayı küçültür.
    Asıl çıktı dosyasını değiştirmez.
    """
    height, width = image.shape[:2]
    longest_side = max(width, height)

    if longest_side <= max_side:
        return image

    ratio = max_side / longest_side

    new_width = max(1, int(width * ratio))
    new_height = max(1, int(height * ratio))

    return cv2.resize(
        image,
        (new_width, new_height),
        interpolation=cv2.INTER_AREA
    )


def visible_rgb_and_alpha(rgba):
    rgb = rgba[:, :, :3]
    alpha = rgba[:, :, 3]
    visible_mask = alpha > 20

    return rgb, alpha, visible_mask


def find_original_file(upscaled_path):
    """
    *_upscaled dosyasına karşılık gelen kaynak görseli bulur.
    """
    directory = os.path.dirname(upscaled_path)
    filename = os.path.basename(upscaled_path)

    stem = filename.rsplit("_upscaled", 1)[0]

    extensions = [".png", ".jpg", ".jpeg", ".webp"]

    candidates = [
        os.path.join(directory, stem + extension)
        for extension in extensions
    ]

    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate

    return None


def canonical_design_key(path):
    """
    Aynı görselin Colab tarafından oluşturulan kopyalarını tek tasarım sayar.

    Örnek:
    wolf_rembg_upscaled.png
    wolf_rembg (1)_upscaled.png

    Bu iki dosya aynı tasarım kabul edilir.
    """
    filename = os.path.basename(path).lower()

    filename = re.sub(
        r"\.(png|jpg|jpeg|webp)$",
        "",
        filename
    )

    filename = filename.replace("_upscaled", "")

    # Dosya sonundaki (1), (2), (3) gibi kopya numaralarını kaldır
    filename = re.sub(
        r"\s*\(\d+\)\s*$",
        "",
        filename
    )

    # "_rembg (1)" biçimini "_rembg" yap
    filename = re.sub(
        r"\s*\(\d+\)(?=_rembg|$)",
        "",
        filename
    )

    # Fazla boşlukları temizle
    filename = re.sub(r"\s+", " ", filename).strip()

    return filename


def remove_duplicate_outputs(paths):
    """
    Aynı tasarımın birden fazla kopyası varsa yalnızca birini seçer.

    Öncelik:
    1. Adında sonradan eklenmiş (1), (2) bulunmayan dosya
    2. Daha yeni dosya
    """
    groups = {}

    for path in paths:
        key = canonical_design_key(path)
        groups.setdefault(key, []).append(path)

    selected = []

    for key, files in groups.items():

        def selection_priority(path):
            stem = os.path.basename(path).rsplit("_upscaled", 1)[0]

            has_copy_suffix = bool(
                re.search(r"\s*\(\d+\)\s*$", stem)
            )

            modification_time = os.path.getmtime(path)

            return (
                has_copy_suffix,
                -modification_time
            )

        files = sorted(files, key=selection_priority)
        selected.append(files[0])

    return sorted(selected)


# ============================================================
# 1. BULANIKLIK VE KESKİNLİK
# ============================================================

def analyze_sharpness(rgba):
    rgb, alpha, visible_mask = visible_rgb_and_alpha(rgba)

    if not np.any(visible_mask):
        return {
            "value": 0.0,
            "status": "FAIL",
            "message": "Görselde analiz edilecek görünür içerik bulunamadı."
        }

    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    gray_masked = gray.copy()
    gray_masked[~visible_mask] = 0

    laplacian = cv2.Laplacian(
        gray_masked,
        cv2.CV_64F
    )

    sharpness = float(
        np.var(laplacian[visible_mask])
    )

    if sharpness < BLUR_FAIL:
        status = "FAIL"
        message = (
            "Görsel belirgin şekilde bulanık veya aşırı yumuşak görünüyor."
        )

    elif sharpness < BLUR_WARNING:
        status = "WARN"
        message = (
            "Keskinlik orta seviyede. Büyük baskıdan önce yakın kontrol önerilir."
        )

    else:
        status = "PASS"
        message = (
            "Keskinlik baskı için yeterli görünüyor."
        )

    return {
        "value": sharpness,
        "status": status,
        "message": message
    }


# ============================================================
# 2. KOYU ALAN VE TON KAYBI
# ============================================================

def analyze_brightness(rgba):
    rgb, alpha, visible_mask = visible_rgb_and_alpha(rgba)

    gray = cv2.cvtColor(
        rgb,
        cv2.COLOR_RGB2GRAY
    )

    visible_gray = gray[visible_mask]

    if visible_gray.size == 0:
        return {
            "mean": 0.0,
            "dark_percent": 100.0,
            "black_clip": 100.0,
            "white_clip": 0.0,
            "status": "FAIL",
            "message": "Görselde görünür içerik bulunamadı."
        }

    mean_brightness = float(
        np.mean(visible_gray)
    )

    dark_percent = safe_percent(
        np.mean(visible_gray < 45) * 100
    )

    black_clip = safe_percent(
        np.mean(visible_gray < 8) * 100
    )

    white_clip = safe_percent(
        np.mean(visible_gray > 247) * 100
    )

    messages = []

    if dark_percent > DARK_FAIL_PERCENT:
        status = "WARN"
        messages.append(
            "Görsel oldukça koyu. Siyah tişörtte bazı ayrıntılar kaybolabilir."
        )

    elif dark_percent > DARK_WARNING_PERCENT:
        status = "WARN"
        messages.append(
            "Koyu alanlar fazla. Özellikle siyah tişört ön izlemesini kontrol edin."
        )

    else:
        status = "PASS"

    if black_clip > DARK_CLIP_WARNING_PERCENT:
        status = "WARN"
        messages.append(
            "Bazı çok koyu bölgelerde ton ayrıntısı az olabilir."
        )

    if white_clip > LIGHT_CLIP_WARNING_PERCENT:
        status = "WARN"
        messages.append(
            "Bazı çok açık bölgelerde ayrıntı az olabilir."
        )

    if not messages:
        message = (
            "Parlaklık ve koyu alan dağılımı uygun görünüyor."
        )
    else:
        message = " ".join(messages)

    return {
        "mean": mean_brightness,
        "dark_percent": dark_percent,
        "black_clip": black_clip,
        "white_clip": white_clip,
        "status": status,
        "message": message
    }


# ============================================================
# 3. ŞEFFAF ARKA PLAN KONTROLÜ
# ============================================================

def analyze_transparency(rgba):
    rgb, alpha, visible_mask = visible_rgb_and_alpha(rgba)

    total_pixels = alpha.size

    fully_transparent = alpha == 0
    semi_transparent = (alpha > 0) & (alpha < 245)

    transparent_percent = safe_percent(
        np.sum(fully_transparent)
        / total_pixels
        * 100
    )

    semi_percent = safe_percent(
        np.sum(semi_transparent)
        / total_pixels
        * 100
    )

    height, width = alpha.shape

    border_width = max(
        2,
        int(min(height, width) * 0.01)
    )

    border = np.concatenate([
        alpha[:border_width, :].flatten(),
        alpha[-border_width:, :].flatten(),
        alpha[:, :border_width].flatten(),
        alpha[:, -border_width:].flatten()
    ])

    border_residue = safe_percent(
        np.mean(border > 10) * 100
    )

    # Kenarda tasarım bulunması her zaman arka plan hatası değildir.
    # Bu nedenle otomatik olarak kesin FAIL verilmez.
    if border_residue > TRANSPARENT_RESIDUE_STRONG:
        status = "WARN"
        message = (
            "Tasarım dış kenarlara çok yaklaşıyor veya kenarda şeffaf kalıntı olabilir. "
            "Koyu ön izlemede kontrol edin."
        )

    elif border_residue > TRANSPARENT_RESIDUE_WARNING:
        status = "INFO"
        message = (
            "Dış kenarlarda görünür piksel bulundu. "
            "Bu, suluboya detayı veya tasarımın doğal parçası olabilir."
        )

    else:
        status = "PASS"
        message = (
            "Şeffaf dış alan temiz görünüyor."
        )

    return {
        "transparent_percent": transparent_percent,
        "semi_percent": semi_percent,
        "border_residue": border_residue,
        "status": status,
        "message": message
    }


# ============================================================
# 4. BEYAZ VE SİYAH KENAR HALOSU
# ============================================================

def analyze_halo(rgba):
    rgb, alpha, visible_mask = visible_rgb_and_alpha(rgba)

    edge_mask = (
        (alpha > 10)
        & (alpha < 245)
    )

    edge_count = int(
        np.sum(edge_mask)
    )

    if edge_count < 20:
        return {
            "white_halo": 0.0,
            "dark_halo": 0.0,
            "status": "PASS",
            "message": "Belirgin yarı şeffaf kenar bulunmadı."
        }

    edge_rgb = rgb[edge_mask].astype(
        np.float32
    )

    brightness = np.mean(
        edge_rgb,
        axis=1
    )

    white_halo = safe_percent(
        np.mean(brightness > 235) * 100
    )

    dark_halo = safe_percent(
        np.mean(brightness < 18) * 100
    )

    strongest_halo = max(
        white_halo,
        dark_halo
    )

    # Halo ölçümü sanatsal beyaz/siyah çizgileri yanlış yorumlayabilir.
    # Bu nedenle yalnızca bilgi veya uyarı verir; kesin hata vermez.
    if strongest_halo > 60:
        status = "INFO"
        message = (
            "Yarı şeffaf kenarlarda açık veya koyu renk yoğunluğu yüksek. "
            "Bu, çizim stilinin doğal parçası olabilir."
        )

    elif strongest_halo > HALO_WARNING:
        status = "INFO"
        message = (
            "Kenar çevresinde hafif açık veya koyu renk riski var. "
            "Açık ve koyu ön izlemede kontrol edin."
        )

    else:
        status = "PASS"
        message = (
            "Belirgin kenar halosu tespit edilmedi."
        )

    return {
        "white_halo": white_halo,
        "dark_halo": dark_halo,
        "status": status,
        "message": message
    }


# ============================================================
# 5. KÜÇÜK VE İNCE DETAY
# ============================================================

def analyze_tiny_details(rgba):
    rgb, alpha, visible_mask = visible_rgb_and_alpha(rgba)

    gray = cv2.cvtColor(
        rgb,
        cv2.COLOR_RGB2GRAY
    )

    edges = cv2.Canny(
        gray,
        80,
        180
    )

    edges[~visible_mask] = 0

    visible_count = max(
        1,
        int(np.sum(visible_mask))
    )

    edge_density = safe_percent(
        np.sum(edges > 0)
        / visible_count
        * 100
    )

    if edge_density > TINY_DETAIL_WARNING:
        status = "WARN"
        message = (
            "Tasarımda çok sayıda ince ayrıntı var. "
            "Çok küçük baskı kullanmayın."
        )

    else:
        status = "PASS"
        message = (
            "İnce detay yoğunluğu güvenli seviyede görünüyor."
        )

    return {
        "value": edge_density,
        "status": status,
        "message": message
    }


# ============================================================
# 6. RENK VE DOYGUNLUK
# ============================================================

def analyze_colors(rgba):
    rgb, alpha, visible_mask = visible_rgb_and_alpha(rgba)

    hsv = cv2.cvtColor(
        rgb,
        cv2.COLOR_RGB2HSV
    )

    saturation = hsv[:, :, 1]

    visible_saturation = saturation[
        visible_mask
    ]

    if visible_saturation.size == 0:
        return {
            "mean_saturation": 0.0,
            "extreme_saturation": 0.0,
            "status": "FAIL",
            "message": "Renk analizi için görünür piksel bulunamadı."
        }

    mean_saturation = float(
        np.mean(visible_saturation)
    )

    extreme_saturation = safe_percent(
        np.mean(visible_saturation > 245)
        * 100
    )

    if extreme_saturation > 30:
        status = "WARN"
        message = (
            "Çok parlak ve doygun renkler fazla. "
            "Fiziksel baskı ekrandan daha mat görünebilir."
        )

    else:
        status = "PASS"
        message = (
            "Renk yoğunluğu baskı için güvenli görünüyor. "
            "Fiziksel baskı ekrandan biraz daha mat olabilir."
        )

    return {
        "mean_saturation": mean_saturation,
        "extreme_saturation": extreme_saturation,
        "status": status,
        "message": message
    }


# ============================================================
# 7. KAYNAK GÖRSELE GÖRE DEĞİŞİM
# ============================================================

def analyze_structure_change(original_path, output_rgba):
    if original_path is None:
        return {
            "difference": None,
            "status": "INFO",
            "message": (
                "Kaynak dosya bulunamadığı için kaynak karşılaştırması yapılmadı."
            )
        }

    if not os.path.exists(original_path):
        return {
            "difference": None,
            "status": "INFO",
            "message": (
                "Kaynak dosya bulunamadığı için kaynak karşılaştırması yapılmadı."
            )
        }

    original_rgba = load_rgba(
        original_path
    )

    original_rgb = original_rgba[:, :, :3]
    output_rgb = output_rgba[:, :, :3]

    original_height, original_width = (
        original_rgb.shape[:2]
    )

    resized_output = cv2.resize(
        output_rgb,
        (original_width, original_height),
        interpolation=cv2.INTER_AREA
    )

    original_gray = cv2.cvtColor(
        original_rgb,
        cv2.COLOR_RGB2GRAY
    )

    output_gray = cv2.cvtColor(
        resized_output,
        cv2.COLOR_RGB2GRAY
    )

    difference = float(
        np.mean(
            np.abs(
                original_gray.astype(np.float32)
                - output_gray.astype(np.float32)
            )
        )
    )

    if difference > STRUCTURE_FAIL:
        status = "FAIL"
        message = (
            "Upscale sonrasında kaynak görsele göre güçlü değişim tespit edildi."
        )

    elif difference > STRUCTURE_WARNING:
        status = "WARN"
        message = (
            "Upscale sonrasında bazı ton veya detay değişimleri olabilir."
        )

    else:
        status = "PASS"
        message = (
            "Ana şekiller kaynak görsele yakın şekilde korunmuş görünüyor."
        )

    return {
        "difference": difference,
        "status": status,
        "message": message
    }


# ============================================================
# 8. PRINTIFY DOSYA VE BOYUT KONTROLÜ
# ============================================================

def analyze_print_requirements(path, rgba):
    height, width = rgba.shape[:2]

    file_size_bytes = os.path.getsize(
        path
    )

    file_size_mb = (
        file_size_bytes
        / (1024 ** 2)
    )

    width_inches = width / TARGET_DPI
    height_inches = height / TARGET_DPI

    problems = []

    if file_size_mb > MAX_PRINTIFY_SIZE_MB:
        problems.append(
            "Dosya 100 MB sınırını aşıyor."
        )

    if min(width, height) < 3000:
        problems.append(
            "Kısa kenar 3000 pikselin altında."
        )

    if file_size_mb > MAX_PRINTIFY_SIZE_MB:
        status = "FAIL"

    elif min(width, height) < 2000:
        status = "FAIL"

    elif problems:
        status = "WARN"

    else:
        status = "PASS"

    if problems:
        message = " ".join(problems)

    else:
        message = (
            "Piksel boyutu ve dosya büyüklüğü Printify için uygun görünüyor."
        )

    return {
        "width": width,
        "height": height,
        "file_size_mb": file_size_mb,
        "file_size_text": human_size(file_size_bytes),
        "width_inches": width_inches,
        "height_inches": height_inches,
        "status": status,
        "message": message
    }


# ============================================================
# DURUM VE PUANLAMA
# ============================================================

def status_icon(status):
    icons = {
        "PASS": "✅",
        "INFO": "ℹ️",
        "WARN": "⚠️",
        "FAIL": "❌"
    }

    return icons.get(status, "ℹ️")


def calculate_final_result(checks):
    """
    INFO sonuçları kalite puanını düşürmez.
    Halo gibi sanatsal kontroller kesin hata sayılmaz.
    """
    fail_count = sum(
        check["status"] == "FAIL"
        for check in checks
    )

    warning_count = sum(
        check["status"] == "WARN"
        for check in checks
    )

    if fail_count >= 2:
        return {
            "code": "FAIL",
            "title": "❌ YENİDEN İŞLE",
            "score": 45,
            "message": (
                "Birden fazla önemli teknik sorun bulundu. "
                "Bu dosyayı hemen baskıya göndermeyin."
            )
        }

    if fail_count == 1:
        return {
            "code": "CHECK",
            "title": "⚠️ KONTROL ET",
            "score": 68,
            "message": (
                "Bir önemli teknik risk bulundu. "
                "Belirtilen noktayı kontrol edin."
            )
        }

    if warning_count >= 3:
        return {
            "code": "CHECK",
            "title": "⚠️ KONTROL ET",
            "score": 72,
            "message": (
                "Birden fazla küçük teknik risk bulundu. "
                "Printify ön izlemesini kontrol edin."
            )
        }

    if warning_count == 2:
        return {
            "code": "READY_CHECK",
            "title": "✅ BASKIYA UYGUN — KÜÇÜK KONTROL",
            "score": 86,
            "message": (
                "Ciddi sorun bulunmadı. "
                "İki küçük noktayı ön izlemede kontrol edin."
            )
        }

    if warning_count == 1:
        return {
            "code": "READY_CHECK",
            "title": "✅ BASKIYA UYGUN — KÜÇÜK KONTROL",
            "score": 94,
            "message": (
                "Ciddi sorun bulunmadı. "
                "Tek küçük uyarıyı ön izlemede kontrol edin."
            )
        }

    return {
        "code": "READY",
        "title": "✅ BASKIYA HAZIR",
        "score": 100,
        "message": (
            "Otomatik teknik kontrollerde belirgin sorun bulunmadı."
        )
    }


# ============================================================
# TEK GÖRSELİ ANALİZ ET
# ============================================================

def analyze_one_image(output_path):
    original_path = find_original_file(
        output_path
    )

    rgba_full = load_rgba(
        output_path
    )

    rgba_analysis = resize_for_analysis(
        rgba_full
    )

    sharpness = analyze_sharpness(
        rgba_analysis
    )

    brightness = analyze_brightness(
        rgba_analysis
    )

    transparency = analyze_transparency(
        rgba_analysis
    )

    halo = analyze_halo(
        rgba_analysis
    )

    tiny_details = analyze_tiny_details(
        rgba_analysis
    )

    colors = analyze_colors(
        rgba_analysis
    )

    structure = analyze_structure_change(
        original_path,
        rgba_full
    )

    print_info = analyze_print_requirements(
        output_path,
        rgba_full
    )

    checks = [
        {
            "name": "Keskinlik ve bulanıklık",
            "status": sharpness["status"],
            "message": sharpness["message"],
            "value": (
                f"Keskinlik puanı: {sharpness['value']:.1f}"
            )
        },
        {
            "name": "Koyu alan ve ton kaybı",
            "status": brightness["status"],
            "message": brightness["message"],
            "value": (
                f"Koyu alan: %{brightness['dark_percent']:.1f} | "
                f"Tam siyaha yakın: %{brightness['black_clip']:.1f}"
            )
        },
        {
            "name": "Şeffaf arka plan temizliği",
            "status": transparency["status"],
            "message": transparency["message"],
            "value": (
                f"Dış kenar görünür piksel: "
                f"%{transparency['border_residue']:.2f}"
            )
        },
        {
            "name": "Kenar rengi ve halo kontrolü",
            "status": halo["status"],
            "message": halo["message"],
            "value": (
                f"Açık kenar: %{halo['white_halo']:.1f} | "
                f"Koyu kenar: %{halo['dark_halo']:.1f}"
            )
        },
        {
            "name": "İnce ve küçük detay riski",
            "status": tiny_details["status"],
            "message": tiny_details["message"],
            "value": (
                f"Detay yoğunluğu: %{tiny_details['value']:.1f}"
            )
        },
        {
            "name": "Renk ve doygunluk",
            "status": colors["status"],
            "message": colors["message"],
            "value": (
                f"Aşırı doygun alan: "
                f"%{colors['extreme_saturation']:.1f}"
            )
        },
        {
            "name": "Kaynak görsele göre değişim",
            "status": structure["status"],
            "message": structure["message"],
            "value": (
                "Ölçülemedi"
                if structure["difference"] is None
                else (
                    f"Değişim puanı: "
                    f"{structure['difference']:.1f}"
                )
            )
        },
        {
            "name": "Printify dosya ve boyut kontrolü",
            "status": print_info["status"],
            "message": print_info["message"],
            "value": (
                f"{print_info['width']} × "
                f"{print_info['height']} px | "
                f"{print_info['file_size_text']}"
            )
        }
    ]

    final = calculate_final_result(
        checks
    )

    print()
    print("=" * 82)
    print("🐺 ORIONINSIST — OTOMATİK BASKI KALİTESİ ANALİZİ")
    print("=" * 82)

    print(
        f"\n📄 Görsel: "
        f"{os.path.basename(output_path)}"
    )

    if original_path:
        print(
            f"📁 Kaynak: "
            f"{os.path.basename(original_path)}"
        )
    else:
        print(
            "📁 Kaynak: Bulunamadı"
        )

    print(
        f"📐 Boyut: "
        f"{print_info['width']} × "
        f"{print_info['height']} px"
    )

    print(
        f"🖨️ 300 DPI baskı alanı: "
        f"{print_info['width_inches']:.2f} × "
        f"{print_info['height_inches']:.2f} inç"
    )

    print("\n🔍 KONTROL SONUÇLARI")

    for check in checks:
        print()
        print(
            f"{status_icon(check['status'])} "
            f"{check['name']}"
        )
        print(
            f"   Sonuç : {check['message']}"
        )
        print(
            f"   Ölçüm : {check['value']}"
        )

    print("\n" + "-" * 82)
    print(
        f"🎯 SON KARAR: {final['title']}"
    )
    print(
        f"📊 Tahmini teknik kalite puanı: "
        f"{final['score']}/100"
    )
    print(
        f"💬 Açıklama: {final['message']}"
    )

    face_restore_is_open = (
        "use_face_restore" in globals()
        and bool(globals()["use_face_restore"])
    )

    if face_restore_is_open:
        print()
        print(
            "❌ ÖNEMLİ: Face Restore açık."
        )
        print(
            "   Kurt, hayvan ve illüstrasyon tasarımları için "
            "Face Restore kapalı olmalıdır."
        )
        print(
            "   use_face_restore = False yaparak yeniden upscale edin."
        )

    print(
        "\n🧑‍💻 SADE KULLANICI ÖZETİ"
    )

    fail_messages = [
        check["message"]
        for check in checks
        if check["status"] == "FAIL"
    ]

    warning_messages = [
        check["message"]
        for check in checks
        if check["status"] == "WARN"
    ]

    info_messages = [
        check["message"]
        for check in checks
        if check["status"] == "INFO"
    ]

    if final["code"] == "FAIL":
        print(
            "❌ BU GÖRSELİ ŞİMDİ BASKIYA GÖNDERME."
        )

        for message in fail_messages:
            print(f"• {message}")

    elif final["code"] == "CHECK":
        print(
            "⚠️ GÖRSEL KULLANILABİLİR OLABİLİR, "
            "AMA ÖNCE UYARILARI KONTROL ET."
        )

        for message in fail_messages + warning_messages:
            print(f"• {message}")

    elif final["code"] == "READY_CHECK":
        print(
            "✅ GÖRSEL BÜYÜK İHTİMALLE BASKIYA UYGUN."
        )

        if warning_messages:
            print(
                "Sadece şu küçük noktaları ön izlemede kontrol et:"
            )

            for message in warning_messages:
                print(f"• {message}")

    else:
        print(
            "✅ GÖRSEL PRINTIFY'A YÜKLENMEYE HAZIR GÖRÜNÜYOR."
        )

    if info_messages:
        print()
        print(
            "ℹ️ Bilgi amaçlı notlar:"
        )

        for message in info_messages:
            print(f"• {message}")

    print()
    print(
        "ℹ️ Fiziksel baskı rengi, ekran görüntüsünden "
        "biraz daha mat olabilir."
    )
    print("=" * 82)

    # Açık ve koyu zemin ön izlemesi
    display(
        HTML(
            f"""
            <div style="
                display:flex;
                gap:20px;
                flex-wrap:wrap;
                margin:20px 0 45px 0;
            ">

                <div style="
                    flex:1;
                    min-width:300px;
                    padding:20px;
                    background:#ffffff;
                    border:1px solid #bbbbbb;
                    text-align:center;
                    box-sizing:border-box;
                ">

                    <div style="
                        margin-bottom:12px;
                        font-weight:bold;
                        color:#111111;
                    ">
                        Açık tişört ön izlemesi
                    </div>

                    <img
                        src="{output_path}"
                        style="
                            max-width:100%;
                            max-height:700px;
                            object-fit:contain;
                        "
                    >

                </div>

                <div style="
                    flex:1;
                    min-width:300px;
                    padding:20px;
                    background:#171717;
                    border:1px solid #444444;
                    text-align:center;
                    box-sizing:border-box;
                ">

                    <div style="
                        margin-bottom:12px;
                        font-weight:bold;
                        color:#ffffff;
                    ">
                        Koyu tişört ön izlemesi
                    </div>

                    <img
                        src="{output_path}"
                        style="
                            max-width:100%;
                            max-height:700px;
                            object-fit:contain;
                        "
                    >

                </div>

            </div>
            """
        )
    )

    return {
        "file": output_path,
        "checks": checks,
        "final": final
    }


# ============================================================
# BÜTÜN ÇIKTILARI BUL
# ============================================================

patterns = [
    "*_upscaled.png",
    "*_upscaled.jpg",
    "*_upscaled.jpeg",
    "*_upscaled.webp"
]

all_output_files = []

for pattern in patterns:
    all_output_files.extend(
        glob.glob(pattern)
    )

all_output_files = sorted(
    set(all_output_files)
)

# Aynı tasarımın (1), (2) gibi kopyalarını kaldır
output_files = remove_duplicate_outputs(
    all_output_files
)


# ============================================================
# ANALİZİ BAŞLAT
# ============================================================

if not output_files:
    print(
        "❌ Analiz edilecek büyütülmüş görsel bulunamadı."
    )
    print(
        "Önce upscale işlemini çalıştırın."
    )

else:
    skipped_count = (
        len(all_output_files)
        - len(output_files)
    )

    print(
        f"🔬 Toplam {len(output_files)} farklı tasarım analiz edilecek."
    )

    if skipped_count > 0:
        print(
            f"ℹ️ {skipped_count} tekrar dosya analiz dışında bırakıldı."
        )

    all_results = []

    for image_path in output_files:
        try:
            result = analyze_one_image(
                image_path
            )

            all_results.append(
                result
            )

        except Exception as error:
            print()
            print("=" * 82)
            print(
                f"❌ Analiz başarısız: "
                f"{os.path.basename(image_path)}"
            )
            print(
                f"Hata: {error}"
            )
            print("=" * 82)


    # ========================================================
    # SADE TOPLU SONUÇ
    # ========================================================

    print()
    print("=" * 82)
    print(
        "📋 BÜTÜN GÖRSELLERİN SADE SONUÇ ÖZETİ"
    )
    print("=" * 82)

    ready_count = 0
    ready_check_count = 0
    warning_count = 0
    failed_count = 0

    for index, result in enumerate(
        all_results,
        start=1
    ):
        final = result["final"]

        print()
        print(
            f"{index}. "
            f"{os.path.basename(result['file'])}"
        )

        print(
            f"   {final['title']} — "
            f"{final['score']}/100"
        )

        # DÜZELTİLMİŞ SINIFLANDIRMA
        # Artık başlıktaki KONTROL kelimesine bakmıyor.
        # Doğrudan güvenli durum kodunu kullanıyor.

        if final["code"] == "FAIL":
            failed_count += 1

        elif final["code"] == "CHECK":
            warning_count += 1

        elif final["code"] == "READY_CHECK":
            ready_check_count += 1

        elif final["code"] == "READY":
            ready_count += 1


    print()
    print("-" * 82)

    print(
        f"✅ Tam baskıya hazır             : {ready_count}"
    )

    print(
        f"✅ Baskıya uygun, küçük kontrol  : {ready_check_count}"
    )

    print(
        f"⚠️ Kontrol edilmesi gereken     : {warning_count}"
    )

    print(
        f"❌ Yeniden işlenecek            : {failed_count}"
    )

    print(
        f"📊 Analiz edilen farklı tasarım : {len(all_results)}"
    )

    if skipped_count > 0:
        print(
            f"♻️ Atlanan tekrar dosya          : {skipped_count}"
        )

    print()
    print("ÖNEMLİ SON KURAL:")

    face_restore_is_open = (
        "use_face_restore" in globals()
        and bool(globals()["use_face_restore"])
    )

    if face_restore_is_open:
        print(
            "❌ Face Restore açık."
        )
        print(
            "Kurt ve illüstrasyon görsellerini "
            "use_face_restore = False ile yeniden üret."
        )

    else:
        print(
            "✅ Face Restore kapalı veya kullanılmıyor."
        )

    print("=" * 82)

🔬 Toplam 4 farklı tasarım analiz edilecek.
ℹ️ 4 tekrar dosya analiz dışında bırakıldı.

🐺 ORIONINSIST — OTOMATİK BASKI KALİTESİ ANALİZİ

📄 Görsel: 2026-07-29_rustic-outdoor_geometric-wolf-badge_001 (2)_rembg_upscaled.png
📁 Kaynak: 2026-07-29_rustic-outdoor_geometric-wolf-badge_001 (2)_rembg.png
📐 Boyut: 5016 × 5016 px
🖨️ 300 DPI baskı alanı: 16.72 × 16.72 inç

🔍 KONTROL SONUÇLARI

✅ Keskinlik ve bulanıklık
   Sonuç : Keskinlik baskı için yeterli görünüyor.
   Ölçüm : Keskinlik puanı: 616.1

✅ Koyu alan ve ton kaybı
   Sonuç : Parlaklık ve koyu alan dağılımı uygun görünüyor.
   Ölçüm : Koyu alan: %10.1 | Tam siyaha yakın: %0.0

✅ Şeffaf arka plan temizliği
   Sonuç : Şeffaf dış alan temiz görünüyor.
   Ölçüm : Dış kenar görünür piksel: %0.00

ℹ️ Kenar rengi ve halo kontrolü
   Sonuç : Kenar çevresinde hafif açık veya koyu renk riski var. Açık ve koyu ön izlemede kontrol edin.
   Ölçüm : Açık kenar: %2.1 | Koyu kenar: %13.6

✅ İnce ve küçük detay riski
   Sonuç : İnce detay yoğunluğu güv


🐺 ORIONINSIST — OTOMATİK BASKI KALİTESİ ANALİZİ

📄 Görsel: 2026-07-29_vintage-wolf_full-moon-retro-graphic-tshirt_001_rembg_upscaled.png
📁 Kaynak: 2026-07-29_vintage-wolf_full-moon-retro-graphic-tshirt_001_rembg.png
📐 Boyut: 4488 × 5608 px
🖨️ 300 DPI baskı alanı: 14.96 × 18.69 inç

🔍 KONTROL SONUÇLARI

✅ Keskinlik ve bulanıklık
   Sonuç : Keskinlik baskı için yeterli görünüyor.
   Ölçüm : Keskinlik puanı: 925.4

✅ Koyu alan ve ton kaybı
   Sonuç : Parlaklık ve koyu alan dağılımı uygun görünüyor.
   Ölçüm : Koyu alan: %28.3 | Tam siyaha yakın: %0.0

✅ Şeffaf arka plan temizliği
   Sonuç : Şeffaf dış alan temiz görünüyor.
   Ölçüm : Dış kenar görünür piksel: %0.00

ℹ️ Kenar rengi ve halo kontrolü
   Sonuç : Kenar çevresinde hafif açık veya koyu renk riski var. Açık ve koyu ön izlemede kontrol edin.
   Ölçüm : Açık kenar: %0.0 | Koyu kenar: %15.7

✅ İnce ve küçük detay riski
   Sonuç : İnce detay yoğunluğu güvenli seviyede görünüyor.
   Ölçüm : Detay yoğunluğu: %6.6

✅ Renk ve doygunluk



🐺 ORIONINSIST — OTOMATİK BASKI KALİTESİ ANALİZİ

📄 Görsel: 2026-07-29_wilderness-tee_rustic-pine-forest-wolf_001 (2)_rembg_upscaled.png
📁 Kaynak: 2026-07-29_wilderness-tee_rustic-pine-forest-wolf_001 (2)_rembg.png
📐 Boyut: 5016 × 5016 px
🖨️ 300 DPI baskı alanı: 16.72 × 16.72 inç

🔍 KONTROL SONUÇLARI

✅ Keskinlik ve bulanıklık
   Sonuç : Keskinlik baskı için yeterli görünüyor.
   Ölçüm : Keskinlik puanı: 28931.4

⚠️ Koyu alan ve ton kaybı
   Sonuç : Koyu alanlar fazla. Özellikle siyah tişört ön izlemesini kontrol edin.
   Ölçüm : Koyu alan: %42.3 | Tam siyaha yakın: %15.1

✅ Şeffaf arka plan temizliği
   Sonuç : Şeffaf dış alan temiz görünüyor.
   Ölçüm : Dış kenar görünür piksel: %0.00

ℹ️ Kenar rengi ve halo kontrolü
   Sonuç : Yarı şeffaf kenarlarda açık veya koyu renk yoğunluğu yüksek. Bu, çizim stilinin doğal parçası olabilir.
   Ölçüm : Açık kenar: %88.3 | Koyu kenar: %0.0

⚠️ İnce ve küçük detay riski
   Sonuç : Tasarımda çok sayıda ince ayrıntı var. Çok küçük baskı kullanmayın.


🐺 ORIONINSIST — OTOMATİK BASKI KALİTESİ ANALİZİ

📄 Görsel: 2026-07-29_wildlife-nature_watercolor-wolf-floral-top_001 (3)_rembg_upscaled.png
📁 Kaynak: 2026-07-29_wildlife-nature_watercolor-wolf-floral-top_001 (3)_rembg.png
📐 Boyut: 5016 × 5016 px
🖨️ 300 DPI baskı alanı: 16.72 × 16.72 inç

🔍 KONTROL SONUÇLARI

✅ Keskinlik ve bulanıklık
   Sonuç : Keskinlik baskı için yeterli görünüyor.
   Ölçüm : Keskinlik puanı: 1575.1

✅ Koyu alan ve ton kaybı
   Sonuç : Parlaklık ve koyu alan dağılımı uygun görünüyor.
   Ölçüm : Koyu alan: %7.4 | Tam siyaha yakın: %0.3

ℹ️ Şeffaf arka plan temizliği
   Sonuç : Dış kenarlarda görünür piksel bulundu. Bu, suluboya detayı veya tasarımın doğal parçası olabilir.
   Ölçüm : Dış kenar görünür piksel: %8.78

✅ Kenar rengi ve halo kontrolü
   Sonuç : Belirgin kenar halosu tespit edilmedi.
   Ölçüm : Açık kenar: %5.0 | Koyu kenar: %10.6

✅ İnce ve küçük detay riski
   Sonuç : İnce detay yoğunluğu güvenli seviyede görünüyor.
   Ölçüm : Detay yoğunluğu: %17.0

✅ 


📋 BÜTÜN GÖRSELLERİN SADE SONUÇ ÖZETİ

1. 2026-07-29_rustic-outdoor_geometric-wolf-badge_001 (2)_rembg_upscaled.png
   ✅ BASKIYA HAZIR — 100/100

2. 2026-07-29_vintage-wolf_full-moon-retro-graphic-tshirt_001_rembg_upscaled.png
   ✅ BASKIYA HAZIR — 100/100

3. 2026-07-29_wilderness-tee_rustic-pine-forest-wolf_001 (2)_rembg_upscaled.png
   ✅ BASKIYA UYGUN — KÜÇÜK KONTROL — 86/100

4. 2026-07-29_wildlife-nature_watercolor-wolf-floral-top_001 (3)_rembg_upscaled.png
   ✅ BASKIYA HAZIR — 100/100

----------------------------------------------------------------------------------
✅ Tam baskıya hazır             : 3
✅ Baskıya uygun, küçük kontrol  : 1
⚠️ Kontrol edilmesi gereken     : 0
❌ Yeniden işlenecek            : 0
📊 Analiz edilen farklı tasarım : 4
♻️ Atlanan tekrar dosya          : 4

ÖNEMLİ SON KURAL:
✅ Face Restore kapalı veya kullanılmıyor.


In [8]:
#@title ⬇️ Tüm sonuç görsellerini ayrı ayrı bilgisayara indir

import os
import glob
import time
from google.colab import files

# Çıktı görsellerini bul
download_patterns = [
    "*_upscaled.png",
    "*_upscaled.jpg",
    "*_upscaled.jpeg",
    "*_upscaled.webp"
]

download_files = []

for pattern in download_patterns:
    download_files.extend(glob.glob(pattern))

# Tekrar edenleri kaldır ve sırala
download_files = sorted(list(set(download_files)))

if not download_files:
    print("❌ İndirilecek büyütülmüş görsel bulunamadı.")
    print("Önce upscale işlemini çalıştırın.")

else:
    print("=" * 75)
    print(f"⬇️ {len(download_files)} GÖRSEL AYRI AYRI İNDİRİLECEK")
    print("=" * 75)

    for index, image_path in enumerate(download_files, start=1):
        if not os.path.exists(image_path):
            print(f"❌ Dosya bulunamadı: {image_path}")
            continue

        file_size_mb = os.path.getsize(image_path) / (1024 ** 2)

        print()
        print(
            f"⬇️ {index}/{len(download_files)} indiriliyor: "
            f"{os.path.basename(image_path)} ({file_size_mb:.2f} MB)"
        )

        try:
            # Dosyayı doğrudan bilgisayara indirir
            files.download(image_path)

            # Tarayıcının arka arkaya indirme komutlarını işlemesi için kısa bekleme
            time.sleep(1.5)

        except Exception as error:
            print(
                f"❌ {os.path.basename(image_path)} indirilemedi: {error}"
            )

    print()
    print("=" * 75)
    print("✅ Bütün indirme komutları gönderildi.")
    print("✅ ZIP oluşturulmadı.")
    print("✅ Her görsel orijinal çıktı kalitesiyle ayrı indirildi.")
    print("=" * 75)

⬇️ 4 GÖRSEL AYRI AYRI İNDİRİLECEK

⬇️ 1/4 indiriliyor: 2026-07-29_rustic-outdoor_geometric-wolf-badge_001 (2)_rembg_upscaled.png (14.27 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


⬇️ 2/4 indiriliyor: 2026-07-29_vintage-wolf_full-moon-retro-graphic-tshirt_001_rembg_upscaled.png (20.06 MB)


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>


⬇️ 4/4 indiriliyor: 2026-07-29_wildlife-nature_watercolor-wolf-floral-top_001 (3)_rembg_upscaled.png (24.95 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Bütün indirme komutları gönderildi.
✅ ZIP oluşturulmadı.
✅ Her görsel orijinal çıktı kalitesiyle ayrı indirildi.
